# Exploración de la cola M/M/1

Este notebook recorre los conceptos del hito 1 de EDsim ejecutando las funciones del paquete
`edsim`: cómo se simula una cola M/M/1, cómo se miden sus métricas, por qué hacen falta réplicas
e intervalos de confianza, y por qué la espera se dispara cerca del 100 % de utilización.

Toda la lógica vive en el paquete y está testeada; aquí solo se llama a sus funciones, se dibuja
y se explica.

## 0. Cómo usar este notebook

- **Kernel:** en Cursor, pulsa *Select Kernel* (arriba a la derecha) y elige el entorno de
  `.venv` del proyecto. Si no aparece, ejecuta antes `uv sync` en la terminal.
- **Ejecutar:** usa *Restart & Run All* (reiniciar y ejecutar todo). El notebook está pensado
  para funcionar de principio a fin sin depender del orden en que ejecutaste las celdas a mano;
  si cambias algo y los resultados te extrañan, reinicia y ejecuta todo.
- **Semillas:** cada sección fija su semilla en una constante visible. Con la misma semilla
  el resultado es idéntico; cámbiala para ver otra muestra del azar.
- **Duración:** en modo normal tarda unos 4 minutos; casi todo es la sección 8 (100
  experimentos de 20 réplicas, dos veces). Para ir más rápido, baja `N_EXPERIMENTOS` en la
  celda de parámetros de abajo.
- **Modo rápido:** con la variable de entorno `EDSIM_NB_RAPIDO=1` todas las réplicas y
  duraciones se reducen y el notebook tarda segundos. Es lo que hace el CI
  (`EDSIM_NB_RAPIDO=1 uv run pytest --nbmake notebooks/`) para comprobar que se ejecuta sin
  errores; los números de ese modo no sirven para sacar conclusiones.
- **Las gráficas desaparecen tras cada commit:** el hook `nbstripout` de pre-commit borra
  resultados e imágenes antes de guardar el notebook en git (se versionan el código y el texto,
  no las salidas). Vuelve a ejecutarlo para verlas.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from edsim.queueing.analysis import (
    interarrival_times,
    occupancy,
    residual_times,
    sample_step,
    service_times,
    step_time_average,
)
from edsim.queueing.mm1 import CustomerRecord, mm1_metrics, simulate_mm1
from edsim.queueing.replications import METRIC_NAMES, compare_mm1, coverage_mm1
from edsim.queueing.report import METRIC_LABELS, format_comparison, format_records, plot_sweep
from edsim.queueing.theory import mm1_theory
from edsim.stats import summarize, t_critical

# --- Parámetros ------------------------------------------------------------------------
# Con EDSIM_NB_RAPIDO=1 (el CI) todo se reduce para que el notebook tarde segundos.
RAPIDO = os.environ.get("EDSIM_NB_RAPIDO") == "1"

N_EXPERIMENTOS = 10 if RAPIDO else 100  # sección 8: experimentos de cobertura
R = 5 if RAPIDO else 20  # réplicas por experimento
DURACION, WARMUP = (1_000.0, 100.0) if RAPIDO else (10_000.0, 1_000.0)
DURACION_CORTA, WARMUP_CORTO = (200.0, 50.0) if RAPIDO else (2_000.0, 500.0)
R_VARIABILIDAD = 10 if RAPIDO else 100  # sección 6
DURACIONES_VARIABILIDAD = [100.0, 400.0, 1_600.0] if RAPIDO else [1_000.0, 4_000.0, 16_000.0]
R_MAX_IC = 20 if RAPIDO else 160  # sección 7
HORIZONTE_MUESTRAS = 5_000.0 if RAPIDO else 50_000.0  # sección 2
N_TRAYECTORIAS = 20 if RAPIDO else 200  # sección 4

plt.rcParams["figure.figsize"] = (9, 4)
print("Modo rápido" if RAPIDO else "Modo normal")

## 1. Introducción: la cola M/M/1

Una **cola M/M/1** es el modelo más sencillo de un servicio con espera: los clientes llegan al
azar, hay **un único servidor** y, si está ocupado, esperan en fila (FIFO: el primero en llegar
es el primero en ser atendido).

La **notación de Kendall** resume el modelo en tres letras, *llegadas / servicio / servidores*:

- **M** (de *Markoviano*): los tiempos entre llegadas son exponenciales, con tasa λ (media 1/λ).
- **M**: los tiempos de servicio son exponenciales, con tasa μ (media 1/μ).
- **1**: un solo servidor.

La **utilización** ρ = λ/μ es la fracción del tiempo que el servidor está ocupado. Si ρ < 1 la
cola alcanza un régimen estable y la teoría da fórmulas cerradas:

| Métrica | Significado | Fórmula |
| --- | --- | --- |
| Wq | espera media en cola | ρ / (μ − λ) |
| W | tiempo medio en el sistema | 1 / (μ − λ) |
| Lq | número medio en cola | ρ² / (1 − ρ) |
| L | número medio en el sistema | ρ / (1 − ρ) |

En la gráfica falta L a propósito: con μ = 1, L = ρ/(1 − ρ) coincide con Wq y las dos curvas
se superpondrían.

El **objetivo** del notebook es comprobar, con la simulación de `edsim`, que estas fórmulas se
cumplen, y entender qué hace falta para que la comparación sea rigurosa.

In [ ]:
teoria = mm1_theory(lam=0.8, mu=1.0)
print(f"λ = 0.8, μ = 1 → ρ = {teoria.rho}")
for nombre in METRIC_NAMES:
    print(f"  {METRIC_LABELS[nombre]:<3} = {getattr(teoria, nombre):.4f}")

In [ ]:
rhos = np.linspace(0.01, 0.95, 200)
curvas = [mm1_theory(r, 1.0) for r in rhos]

fig, ax = plt.subplots()
ax.plot(rhos, [t.wq for t in curvas], label="Wq (espera en cola)")
ax.plot(rhos, [t.w for t in curvas], ":", label="W (tiempo en el sistema)")
ax.plot(rhos, [t.lq for t in curvas], "--", label="Lq (número en cola)")
ax.set_xlabel("Utilización ρ")
ax.set_title("Teoría de la M/M/1 con μ = 1")
ax.set_ylim(0, 20)
ax.grid(alpha=0.3)
ax.legend()
plt.show()

**Para experimentar**

- ¿Cuánto crece Wq al pasar de ρ = 0,8 a 0,9? ¿Y de 0,9 a 0,95? Calcúlalo con `mm1_theory`.
- Con μ = 2 y el mismo ρ, ¿qué métricas cambian y cuáles no?

> **Lo que debes recordar:** una M/M/1 tiene llegadas y servicios exponenciales y un servidor; con ρ = λ/μ < 1 la teoría da Wq, W, Lq y L, que crecen sin límite al acercarse ρ a 1.

## 2. Llegadas y servicio: la distribución exponencial

En el modelo nadie decide cuánto espera un cliente: solo se sortean dos cosas, **cuándo llega**
(tiempo entre llegadas, exponencial de media 1/λ) y **cuánto dura su servicio** (exponencial de
media 1/μ). La espera *emerge* de la cola.

Tres ideas:

- **Tasa frente a media.** La tasa λ = 0,8 significa 0,8 llegadas por unidad de tiempo; la
  media entre llegadas es 1/λ = 1,25. `numpy` pide la **escala** (la media), no la tasa: escribir
  `exponential(lam)` en vez de `exponential(1 / lam)` es un error silencioso que cambia el modelo.
- **Densidad.** La exponencial de tasa λ tiene densidad f(t) = λ·e^(−λt): muchos intervalos
  cortos y pocos largos.
- **Falta de memoria.** Si un servicio ya ha durado s, lo que le queda sigue siendo exponencial
  con la misma media: lo ya transcurrido no informa de lo que falta. Por eso el modelo no
  necesita recordar desde cuándo dura cada servicio.

In [ ]:
SEMILLA = 21
LAM, MU = 0.8, 1.0

registros = simulate_mm1(LAM, MU, horizon=HORIZONTE_MUESTRAS, seed=SEMILLA)
entre_llegadas = interarrival_times(registros)
servicios = service_times(registros)
print(f"{len(registros)} clientes")
print(f"media entre llegadas: {entre_llegadas.mean():.4f}  (1/λ = {1 / LAM:.4f})")
print(f"media de servicio:    {servicios.mean():.4f}  (1/μ = {1 / MU:.4f})")

In [ ]:
t = np.linspace(0, 8, 300)
bins = np.linspace(0, 8, 41)

fig, (ax1, ax2) = plt.subplots(1, 2, sharey=True)
ax1.hist(entre_llegadas, bins=bins, density=True, alpha=0.5, label="simulación")
ax1.plot(t, LAM * np.exp(-LAM * t), label="λ·e^(−λt)")
ax1.set_title("Tiempo entre llegadas (λ = 0,8)")
ax1.set_xlabel("tiempo")
ax1.legend()
ax2.hist(servicios, bins=bins, density=True, alpha=0.5, label="simulación")
ax2.plot(t, MU * np.exp(-MU * t), label="μ·e^(−μt)")
ax2.set_title("Tiempo de servicio (μ = 1)")
ax2.set_xlabel("tiempo")
ax2.legend()
plt.show()

**El error de escala, visto en una gráfica.** El histograma de los tiempos entre llegadas
cuadra con la exponencial de **media 1/λ = 1,25**. Si se hubiera pasado la tasa como escala, la
media sería λ = 0,8: llegarían clientes más a menudo y ρ sería 1,25, una cola inestable.

In [ ]:
fig, ax = plt.subplots()
ax.hist(entre_llegadas, bins=bins, density=True, alpha=0.4, label="simulación de edsim")
ax.plot(t, LAM * np.exp(-LAM * t), label="media 1/λ = 1,25 (correcto)")
ax.plot(t, (1 / LAM) * np.exp(-t / LAM), "--", label="media λ = 0,8 (tasa pasada como escala)")
ax.set_xlabel("tiempo entre llegadas")
ax.set_ylabel("densidad")
ax.legend()
plt.show()

**La falta de memoria.** Tomamos los servicios que han durado más de s y miramos lo que les
queda (`residual_times`). Si la exponencial no tiene memoria, las curvas se superponen con la
original para cualquier s.

In [ ]:
fig, ax = plt.subplots()
ax.hist(servicios, bins=bins, density=True, histtype="step", lw=2, label="todos los servicios")
for transcurrido in (1.0, 2.0):
    resto = residual_times(servicios, transcurrido)
    ax.hist(
        resto,
        bins=bins,
        density=True,
        histtype="step",
        label=f"resto tras s = {transcurrido:g} (n = {len(resto)}, media {resto.mean():.3f})",
    )
ax.plot(t, MU * np.exp(-MU * t), "k:", label="μ·e^(−μt)")
ax.set_xlabel("tiempo")
ax.set_ylabel("densidad")
ax.legend()
plt.show()

**Para experimentar**

- Cambia `transcurrido` a 3 o 4. ¿Sigue igual la curva? ¿Cuántas muestras quedan, y por qué el
  histograma se vuelve más irregular?
- Cambia `SEMILLA`. ¿Cuánto se mueven las medias? ¿Y si reduces `HORIZONTE_MUESTRAS` a 500?

> **Lo que debes recordar:** solo se sortean llegadas (media 1/λ) y servicios (media 1/μ); numpy recibe la media, no la tasa, y la exponencial no tiene memoria: lo que queda no depende de lo ya transcurrido.

## 3. Una réplica paso a paso

Una simulación corta permite seguir a cada cliente. `simulate_mm1` devuelve un **registro por
cliente**: cuándo llegó, qué servicio se le asignó, cuándo empezó a ser atendido y cuándo salió.
Si un evento no ocurrió antes del final, se guarda como vacío (`—`): no se inventa.

El **tiempo de servicio se asigna al llegar**, no al empezar la atención. Así el cliente k
recibe siempre el k-ésimo valor del flujo de servicio, pase lo que pase en el resto del sistema.
Es la base de los **números aleatorios comunes**: al comparar dos escenarios con la misma
semilla, las diferencias se deben al escenario y no al azar.

En el diagrama de Gantt, cada fila es un cliente: la barra gris es su espera y la de color, su
servicio. Debajo, cuándo está ocupado el servidor.

In [ ]:
SEMILLA = 3
HORIZONTE = 20.0

registros = simulate_mm1(0.8, 1.0, horizon=HORIZONTE, seed=SEMILLA)
print(format_records(registros))

In [ ]:
ocupacion = occupancy(registros, horizon=HORIZONTE)

fig, (ax, ax_srv) = plt.subplots(
    2, 1, sharex=True, figsize=(9, 5), gridspec_kw={"height_ratios": [4, 1]}
)
for r in registros:
    inicio = r.service_start if r.service_start is not None else HORIZONTE
    fin = r.departure if r.departure is not None else HORIZONTE
    fila = r.customer_id
    ax.broken_barh([(r.arrival, inicio - r.arrival)], (fila - 0.35, 0.7), color="lightgray")
    ax.broken_barh([(inicio, fin - inicio)], (fila - 0.35, 0.7), color="tab:blue")
    ax.plot(r.arrival, fila, "k|", ms=12)
ax.broken_barh([(0, 0)], (0, 0), color="lightgray", label="espera")
ax.broken_barh([(0, 0)], (0, 0), color="tab:blue", label="servicio")
ax.invert_yaxis()
ax.set_ylabel("cliente")
ax.set_title(f"Gantt de una réplica (ρ = 0,8, semilla {SEMILLA}); | = llegada")
ax.legend(loc="upper right")
ax_srv.stairs(ocupacion.busy, ocupacion.edges, fill=True, color="tab:blue")
ax_srv.set_yticks([0, 1], ["libre", "ocupado"])
ax_srv.set_xlabel("tiempo")
ax_srv.set_xlim(0, HORIZONTE)
plt.show()

**Para experimentar**

- Con la misma `SEMILLA`, cambia λ a 0,5. ¿Cambia la columna de servicio asignado? ¿Y las
  llegadas? ¿Por qué es útil que el servicio no cambie?
- Busca en la tabla un cliente que no espera (espera 0). ¿Cómo se ve en el Gantt?

> **Lo que debes recordar:** cada cliente deja un registro (llegada, servicio asignado, inicio y fin); el servicio se asigna al llegar para que, con la misma semilla, cada cliente tenga el mismo servicio aunque cambie el resto del sistema.

## 4. El sistema en el tiempo

El número de clientes **en cola** y **en el sistema** (en cola más en servicio) es una
**función escalonada**: solo cambia en las llegadas, los inicios de servicio y las salidas.
`occupancy` construye esa escalera a partir de los registros.

La simulación **arranca vacía**, algo poco representativo del régimen estable (con ρ = 0,8 hay
de media L = 4 clientes en el sistema). Las primeras unidades de tiempo están sesgadas hacia
abajo: por eso se descarta un **calentamiento** (*warmup*). Una sola trayectoria es demasiado
ruidosa para ver el sesgo; la media de muchas réplicas en cada instante (`sample_step`) sí lo
muestra.

In [ ]:
SEMILLA = 4
HORIZONTE = 500.0

ocupacion = occupancy(simulate_mm1(0.8, 1.0, horizon=HORIZONTE, seed=SEMILLA), horizon=HORIZONTE)

fig, ax = plt.subplots()
ax.stairs(ocupacion.in_system, ocupacion.edges, label="en el sistema")
ax.stairs(ocupacion.in_queue, ocupacion.edges, label="en cola")
ax.axhline(mm1_theory(0.8, 1.0).l, color="k", ls=":", label="L teórica = 4")
ax.set_xlabel("tiempo")
ax.set_ylabel("clientes")
ax.set_title(f"Una réplica (ρ = 0,8, semilla {SEMILLA})")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots()
for semilla in range(SEMILLA, SEMILLA + 5):
    occ = occupancy(simulate_mm1(0.8, 1.0, horizon=HORIZONTE, seed=semilla), horizon=HORIZONTE)
    ax.stairs(occ.in_system, occ.edges, alpha=0.7, label=f"semilla {semilla}")
ax.set_xlabel("tiempo")
ax.set_ylabel("clientes en el sistema")
ax.set_title("Cinco semillas: el mismo modelo, trayectorias muy distintas")
ax.legend()
plt.show()

In [ ]:
rejilla = np.linspace(0, HORIZONTE, 501)
trayectorias = []
for semilla in range(N_TRAYECTORIAS):
    occ = occupancy(simulate_mm1(0.8, 1.0, horizon=HORIZONTE, seed=semilla), horizon=HORIZONTE)
    trayectorias.append(sample_step(occ.times, occ.in_system, rejilla))
media = np.mean(trayectorias, axis=0)

fig, ax = plt.subplots()
ax.plot(rejilla, media, label=f"media de {N_TRAYECTORIAS} réplicas en cada instante")
ax.axhline(mm1_theory(0.8, 1.0).l, color="k", ls=":", label="L teórica = 4")
ax.set_xlabel("tiempo")
ax.set_ylabel("clientes en el sistema")
ax.set_title("El calentamiento: el sistema arranca vacío y tarda en llegar al régimen estable")
ax.legend()
plt.show()

**Para experimentar**

- ¿Desde qué instante parece estable la media? ¿Es suficiente el warmup de 1000 que usamos en
  los experimentos?
- Repite con ρ = 0,9 (λ = 0,9). ¿Tarda más o menos en estabilizarse? ¿Por qué?

> **Lo que debes recordar:** el número en cola y en el sistema es una escalera que cambia en cada evento; como la simulación arranca vacía, las primeras unidades de tiempo están sesgadas y se descartan como calentamiento.

## 5. Métricas: por cliente y en el tiempo

Hay dos tipos de media, y no se calculan igual:

- **Por cliente** (Wq, W): se promedia sobre los clientes que llegan después del calentamiento y
  terminan antes del final. Los que siguen dentro al final se excluyen, porque no se sabe cuánto
  esperarán.
- **En el tiempo** (Lq, L, ρ): el **área bajo la escalera** dividida entre la longitud de la
  ventana. La estancia de cada cliente se **recorta** a la ventana [warmup, horizonte], y los que
  siguen dentro al final cuentan hasta el horizonte.

Las dos se unen en la **ley de Little**: L = λ·W (y Lq = λ·Wq). Se cumple en cualquier sistema
estable, sea cual sea la distribución de llegadas y servicios.

Para verlo con números pequeños, rehacemos el **ejemplo a mano** de la tarea 1.3: cuatro
clientes y la ventana [2, 10].

In [ ]:
# Ejemplo a mano de la tarea 1.3 (el mismo que usan los tests).
ejemplo = (
    CustomerRecord(customer_id=0, arrival=0.0, service_time=3.0, service_start=0.0, departure=3.0),
    CustomerRecord(customer_id=1, arrival=2.5, service_time=2.0, service_start=3.0, departure=5.0),
    CustomerRecord(customer_id=2, arrival=4.0, service_time=1.0, service_start=5.0, departure=6.0),
    CustomerRecord(customer_id=3, arrival=7.0, service_time=4.0, service_start=7.0, departure=None),
)
INICIO, FIN = 2.0, 10.0

print(format_records(ejemplo))
m = mm1_metrics(ejemplo, warmup=INICIO, horizon=FIN)
occ = occupancy(ejemplo, horizon=FIN)
l_escalera = step_time_average(occ.times, occ.in_system, INICIO, FIN)
print(f"\nWq por cliente (clientes 1 y 2): {m.wq}")
print(f"L con mm1_metrics:        {m.l}")
print(f"L con el área de la escalera: {l_escalera}")

In [ ]:
fig, ax = plt.subplots()
ax.axvspan(INICIO, FIN, color="gold", alpha=0.15, label="ventana [2, 10]")
ax.stairs(occ.in_system, occ.edges, fill=True, alpha=0.5, label="en el sistema")
ax.stairs(occ.in_queue, occ.edges, fill=True, alpha=0.7, label="en cola")
ax.axvline(INICIO, color="k", ls="--", lw=1)
ax.set_xlabel("tiempo")
ax.set_ylabel("clientes")
ax.set_title(f"L = área en la ventana / 8 = {l_escalera:.4f}  (mm1_metrics: {m.l:.4f})")
ax.legend()
plt.show()

**La ley de Little en la simulación.** En cada réplica de un experimento comparamos L con
λ_obs·W, donde λ_obs es la tasa de llegadas observada en la ventana. Si los puntos caen sobre la
diagonal, Little se cumple réplica a réplica (salvo los efectos de borde de la ventana).

In [ ]:
SEMILLA = 5

experimento = compare_mm1(0.8, 1.0, DURACION, WARMUP, replicas=R, seed=SEMILLA)
l_rep = np.array([r.l for r in experimento.replications])
lam_w = np.array([r.lam_obs * r.w for r in experimento.replications])
print(f"Diferencia relativa máxima entre L y λ_obs·W: {np.max(np.abs(l_rep - lam_w) / l_rep):.2%}")

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(lam_w, l_rep)
limites = [min(l_rep.min(), lam_w.min()), max(l_rep.max(), lam_w.max())]
ax.plot(limites, limites, "k:", label="L = λ·W")
ax.set_xlabel("λ_obs · W")
ax.set_ylabel("L")
ax.set_title(f"Ley de Little en {R} réplicas")
ax.legend()
plt.show()

**Para experimentar**

- En el ejemplo a mano, cambia `INICIO` a 3. ¿Qué cliente deja de contar en Wq? ¿Cómo cambia L?
- ¿Por qué no podemos incluir al cliente 3 en Wq, si sí cuenta en L?

> **Lo que debes recordar:** Wq y W son medias por cliente; Lq, L y ρ son áreas bajo la escalera divididas entre el tiempo, recortadas a la ventana; la ley de Little, L = λ·W, une las dos y se cumple en cada réplica.

## 6. Variabilidad entre réplicas

Una réplica es **una muestra** del azar: con otra semilla, la Wq medida es distinta. Para ver
cuánto varía, repetimos muchas réplicas con duraciones distintas.

- **Dispersión.** Con más duración, cada réplica promedia más clientes y su resultado varía
  menos. La desviación típica baja aproximadamente como 1/√duración: cuadruplicar la duración la
  divide entre 2.
- **Sesgo.** Con duraciones cortas pesan más los bordes: el arranque vacío y los clientes
  excluidos por no terminar, que suelen ser los que más esperan. La media sale algo por debajo de
  la teoría.

In [ ]:
SEMILLA = 6
teoria_wq = mm1_theory(0.8, 1.0).wq

por_duracion = {
    d: compare_mm1(0.8, 1.0, d, WARMUP, replicas=R_VARIABILIDAD, seed=SEMILLA)
    for d in DURACIONES_VARIABILIDAD
}

todas = [r.wq for c in por_duracion.values() for r in c.replications]
bins_wq = np.linspace(min(todas), max(todas), 40)

fig, ax = plt.subplots()
for d, c in por_duracion.items():
    wq = [r.wq for r in c.replications]
    ax.hist(wq, bins=bins_wq, alpha=0.5, density=True, label=f"duración {d:g}")
ax.axvline(teoria_wq, color="k", ls=":", label=f"Wq teórica = {teoria_wq:g}")
ax.set_xlabel("Wq de cada réplica")
ax.set_title(f"{R_VARIABILIDAD} réplicas por duración (ρ = 0,8)")
ax.legend()
plt.show()

In [ ]:
duraciones = np.array(DURACIONES_VARIABILIDAD)
desviaciones = np.array([c.metric("wq").summary.std for c in por_duracion.values()])
resumenes = [c.metric("wq").summary for c in por_duracion.values()]

fig, (ax1, ax2) = plt.subplots(1, 2)
ax1.loglog(duraciones, desviaciones, "o-", label="desviación típica de Wq")
ax1.loglog(
    duraciones, desviaciones[0] * np.sqrt(duraciones[0] / duraciones), "k:", label="∝ 1/√duración"
)
ax1.set_xlabel("duración")
ax1.set_title("Dispersión")
ax1.legend()
medias = np.array([s.mean for s in resumenes])
errores = np.array(
    [[s.mean - s.ci_low for s in resumenes], [s.ci_high - s.mean for s in resumenes]]
)
ax2.errorbar(duraciones, medias, yerr=errores, fmt="o", capsize=4, label="media e IC 95 %")
ax2.axhline(teoria_wq, color="k", ls=":", label="teoría")
ax2.set_xscale("log")
ax2.set_xlabel("duración")
ax2.set_title("Sesgo")
ax2.legend()
plt.show()

**Para experimentar**

- ¿Cae la dispersión exactamente como 1/√duración? Prueba otra `SEMILLA` y más réplicas antes
  de sacar conclusiones: la propia desviación típica es una estimación con ruido.
- Si cuadruplicas la duración, ¿por cuánto se divide la desviación típica? ¿Y cuánto cuesta en
  tiempo de cálculo?
- ¿Es mejor una réplica muy larga o muchas cortas? Piensa en el sesgo y en el intervalo de
  confianza (sección 7).

> **Lo que debes recordar:** cada réplica da un valor distinto; más duración reduce la dispersión (como 1/√duración) y el sesgo de los bordes, y por eso un único número simulado no basta: hacen falta réplicas.

## 7. Intervalos de confianza

Con R réplicas independientes tenemos R valores de Wq. Su media estima la Wq verdadera, y el
**intervalo de confianza** dice cuánto podemos fiarnos:

1. **Error estándar:** sem = s / √R, donde s es la desviación típica de las réplicas.
2. **t de Student:** la semiamplitud es t·sem. Se usa la t (y no el 1,96 de la normal) porque s
   se estima con pocas réplicas; con R pequeño, t es claramente mayor que 1,96.
3. **IC al 95 %:** media ± t·sem. Si repitiéramos el experimento muchas veces, el 95 % de los
   intervalos contendría el valor verdadero (lo comprobamos en la sección 8).

Como sem ∝ 1/√R, **para dividir el intervalo entre 2 hacen falta 4 veces más réplicas**. Con
pocas réplicas la semiamplitud baja incluso más deprisa, porque además t se acerca a 1,96;
en la gráfica de la derecha se ve la diferencia entre el error estándar y la semiamplitud.

In [ ]:
SEMILLA = 7

grande = compare_mm1(0.8, 1.0, DURACION_CORTA, WARMUP_CORTO, replicas=R_MAX_IC, seed=SEMILLA)
wq = np.array([r.wq for r in grande.replications])

# Paso a paso con las primeras R réplicas, y la misma cuenta con summarize.
x = wq[:R]
s = x.std(ddof=1)
sem = s / np.sqrt(R)
t = t_critical(R, 0.95)
print(f"media = {x.mean():.4f}, s = {s:.4f}, sem = {sem:.4f}, t({R - 1} g.l.) = {t:.4f}")
print(f"IC a mano:   [{x.mean() - t * sem:.4f}, {x.mean() + t * sem:.4f}]")
resumen = summarize(x)
print(f"summarize:   [{resumen.ci_low:.4f}, {resumen.ci_high:.4f}]")

In [ ]:
# La réplica i solo depende de (semilla, i): las primeras R de un experimento de R_MAX_IC
# réplicas son las mismas que las de un experimento de R réplicas.
tamanos = [r for r in (5, 10, 20, 40, 80, 160) if r <= R_MAX_IC]
resumenes_r = [summarize(wq[:r]) for r in tamanos]
errores_estandar = np.array([s.sem for s in resumenes_r])
semiamplitudes = np.array([(s.ci_high - s.ci_low) / 2 for s in resumenes_r])
n = np.arange(2, 41)

fig, (ax1, ax2) = plt.subplots(1, 2)
ax1.plot(n, [t_critical(k, 0.95) for k in n], label="t de Student (95 %)")
ax1.axhline(1.96, color="k", ls=":", label="normal: 1,96")
ax1.set_xlabel("número de réplicas")
ax1.set_ylim(1.5, 5)
ax1.set_title("Valor crítico")
ax1.legend()
ax2.loglog(tamanos, semiamplitudes, "o-", label="semiamplitud del IC (t·sem)")
ax2.loglog(tamanos, errores_estandar, "s-", label="error estándar (sem)")
ax2.loglog(
    tamanos, errores_estandar[0] * np.sqrt(tamanos[0] / np.array(tamanos)), "k:", label="∝ 1/√R"
)
ax2.set_xlabel("réplicas R")
ax2.set_title("Anchura del intervalo")
ax2.legend()
plt.show()

**Para experimentar**

- ¿Cuántas réplicas hacen falta para dividir el intervalo entre 2? ¿Y entre 10?
- Con R = 2, ¿cuánto vale t? ¿Por qué un intervalo con dos réplicas es tan ancho?

> **Lo que debes recordar:** el IC es media ± t·s/√R; la t de Student compensa estimar s con pocas réplicas, y la anchura baja como 1/√R: cuatro veces más réplicas para la mitad de anchura.

## 8. Cobertura: qué significa "95 %"

El 95 % no es la probabilidad de que *este* intervalo contenga la teoría: es la fracción de
experimentos cuyo intervalo la contiene si repitiéramos el experimento muchas veces. Lo
comprobamos repitiendo el experimento completo (R réplicas) con `N_EXPERIMENTOS` semillas
maestras, 0, 1, 2…, y dibujando cada intervalo como una línea horizontal: un **bosque**.

- Con 100 experimentos, la cobertura esperada es 95 ± 4,4 (dos errores estándar binomiales).
- Los fallos deberían repartirse **a ambos lados** de la teoría. Si caen todos del mismo lado,
  la simulación tiene **sesgo**: con duración corta, los bordes bajan las esperas.

Es el experimento que hicimos durante la tarea 1.4, ahora dentro del paquete (`coverage_mm1`).

In [ ]:
SEMILLAS = range(N_EXPERIMENTOS)

corta = coverage_mm1(0.8, 1.0, DURACION_CORTA, WARMUP_CORTO, R, seeds=SEMILLAS, metric="wq")
larga = coverage_mm1(0.8, 1.0, DURACION, WARMUP, R, seeds=SEMILLAS, metric="wq")

for nombre, cov in [("corta", corta), ("larga", larga)]:
    print(
        f"duración {nombre}: {cov.hits}/{len(cov.seeds)} intervalos contienen la teoría; "
        f"simulación por debajo en {cov.below}, por encima en {cov.above}"
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 7), sharey=True)
for ax, cov, titulo in [
    (axes[0], corta, f"Corta: {DURACION_CORTA:g} + warmup {WARMUP_CORTO:g}"),
    (axes[1], larga, f"Larga: {DURACION:g} + warmup {WARMUP:g}"),
]:
    bajos = [m.summary.ci_low for m in cov.intervals]
    altos = [m.summary.ci_high for m in cov.intervals]
    colores = ["tab:blue" if m.theory_in_ci else "tab:red" for m in cov.intervals]
    ax.hlines(list(cov.seeds), bajos, altos, colors=colores)
    ax.axvline(cov.intervals[0].theory, color="k", ls=":")
    ax.set_title(
        f"{titulo}\n{cov.hits}/{len(cov.seeds)} dentro · "
        f"{len(cov.below)} debajo, {len(cov.above)} encima"
    )
    ax.set_xlabel("IC 95 % de Wq")
axes[0].set_ylabel("semilla maestra")
plt.show()

**Para experimentar**

- ¿De qué lado caen los fallos con duración corta? ¿Y con duración larga? ¿Qué te dice eso del
  sesgo?
- Repite con `confidence=0.90` en `coverage_mm1`. ¿Cuántos fallos esperas ahora?

> **Lo que debes recordar:** un IC al 95 % falla en uno de cada veinte experimentos correctos por puro azar; si los fallos caen todos del mismo lado, el problema es el sesgo, no el azar.

## 9. Utilización y no linealidad

¿Por qué la espera se dispara cerca del 100 % de utilización? Wq = ρ/(μ − λ) tiene el factor
1/(1 − ρ): al pasar de ρ = 0,8 a 0,9 la espera se duplica (4 → 9), y de 0,9 a 0,95 vuelve a
duplicarse (9 → 19). Con poca holgura, cada racha de llegadas forma una cola que el servidor
apenas tiene tiempo de vaciar antes de la siguiente.

Además, cerca de la saturación las colas son largas y muy variables: los intervalos se
ensanchan y hacen falta muchas más réplicas para la misma precisión.

El barrido es el mismo que genera `edsim mm1 barrido`, y la gráfica es la misma función
(`plot_sweep`).

In [ ]:
SEMILLA = 2026
RHOS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]

barrido = [(rho, compare_mm1(rho, 1.0, DURACION, WARMUP, replicas=R, seed=SEMILLA)) for rho in RHOS]
display(plot_sweep(barrido))

In [ ]:
relativas = [
    (c.metric("wq").summary.ci_high - c.metric("wq").summary.ci_low)
    / 2
    / c.metric("wq").summary.mean
    for _, c in barrido
]

fig, ax = plt.subplots()
ax.plot(RHOS, np.array(relativas) * 100, "o-")
ax.set_xlabel("Utilización ρ")
ax.set_ylabel("semiamplitud del IC / media (%)")
ax.set_title("Precisión relativa de Wq frente a ρ (mismas réplicas y duración)")
ax.grid(alpha=0.3)
plt.show()

**Para experimentar**

- ¿Cuántas réplicas harían falta con ρ = 0,95 para tener la precisión relativa de ρ = 0,5?
  (Pista: la anchura baja como 1/√R.)
- ¿Por qué con ρ = 0,1 la precisión relativa también empeora, si la cola casi no existe?

> **Lo que debes recordar:** la espera crece como 1/(1 − ρ), así que se dispara cerca del 100 % de utilización: sin holgura, las rachas de llegadas forman colas que no da tiempo a vaciar, y además los resultados se vuelven mucho más variables.

## 10. Para experimentar

Cambia los parámetros de la celda siguiente y ejecuta las dos celdas. La tabla es la misma que
imprime `edsim mm1 comparar`; la gráfica muestra cada métrica dividida entre su valor teórico
(1 = cuadra con la teoría).

Preguntas guía:

- **ρ = 0,99** (`LAM = 0.99`): ¿cuadra con la teoría con la duración por defecto? ¿Qué harías?
- **Sin calentamiento** (`WARMUP_EXP = 0`): ¿cambian los resultados con duración corta? ¿Y larga?
- **Pocas réplicas** (`R_EXP = 3`): ¿cómo son los intervalos?
- **Duración muy corta** (`DURACION_EXP = 100`): ¿qué se rompe antes, el sesgo o la precisión?
- **μ = 2 y λ = 1,6**: mismo ρ que el caso base. ¿Qué métricas cambian?

In [ ]:
LAM_EXP, MU_EXP = 0.8, 1.0
DURACION_EXP, WARMUP_EXP = DURACION, WARMUP
R_EXP = R
SEMILLA_EXP = 2026

exp = compare_mm1(LAM_EXP, MU_EXP, DURACION_EXP, WARMUP_EXP, replicas=R_EXP, seed=SEMILLA_EXP)
print(format_comparison(exp))

In [ ]:
nombres = [METRIC_LABELS[m.name] for m in exp.metrics]
cocientes = [m.summary.mean / m.theory for m in exp.metrics]
errores = [
    [(m.summary.mean - m.summary.ci_low) / m.theory for m in exp.metrics],
    [(m.summary.ci_high - m.summary.mean) / m.theory for m in exp.metrics],
]

fig, ax = plt.subplots()
ax.errorbar(nombres, cocientes, yerr=errores, fmt="o", capsize=5)
ax.axhline(1, color="k", ls=":")
ax.set_ylabel("simulación / teoría")
ax.set_title(f"λ = {LAM_EXP:g}, μ = {MU_EXP:g}, {R_EXP} réplicas, duración {DURACION_EXP:g}")
plt.show()

> **Lo que debes recordar:** para fiarte de una simulación necesitas un régimen estable, calentamiento, duración suficiente y varias réplicas; si falta alguno, la comparación con la teoría lo delata.

## 11. Resumen

Lo que debes recordar de cada sección:

1. Una M/M/1 tiene llegadas y servicios exponenciales y un servidor; con ρ = λ/μ < 1 la teoría da Wq, W, Lq y L, que crecen sin límite al acercarse ρ a 1.
2. Solo se sortean llegadas (media 1/λ) y servicios (media 1/μ); numpy recibe la media, no la tasa, y la exponencial no tiene memoria: lo que queda no depende de lo ya transcurrido.
3. Cada cliente deja un registro (llegada, servicio asignado, inicio y fin); el servicio se asigna al llegar para que, con la misma semilla, cada cliente tenga el mismo servicio aunque cambie el resto del sistema.
4. El número en cola y en el sistema es una escalera que cambia en cada evento; como la simulación arranca vacía, las primeras unidades de tiempo están sesgadas y se descartan como calentamiento.
5. Wq y W son medias por cliente; Lq, L y ρ son áreas bajo la escalera divididas entre el tiempo, recortadas a la ventana; la ley de Little, L = λ·W, une las dos y se cumple en cada réplica.
6. Cada réplica da un valor distinto; más duración reduce la dispersión (como 1/√duración) y el sesgo de los bordes, y por eso un único número simulado no basta: hacen falta réplicas.
7. El IC es media ± t·s/√R; la t de Student compensa estimar s con pocas réplicas, y la anchura baja como 1/√R: cuatro veces más réplicas para la mitad de anchura.
8. Un IC al 95 % falla en uno de cada veinte experimentos correctos por puro azar; si los fallos caen todos del mismo lado, el problema es el sesgo, no el azar.
9. La espera crece como 1/(1 − ρ), así que se dispara cerca del 100 % de utilización: sin holgura, las rachas de llegadas forman colas que no da tiempo a vaciar, y además los resultados se vuelven mucho más variables.
10. Para fiarte de una simulación necesitas un régimen estable, calentamiento, duración suficiente y varias réplicas; si falta alguno, la comparación con la teoría lo delata.

En dos frases, las que pide el hito: **la espera se dispara cerca del 100 % de utilización** porque crece como 1/(1 − ρ) y sin holgura las colas no se vacían; y **hacen falta réplicas** porque cada simulación es una muestra del azar, y solo con varias réplicas independientes se puede construir un intervalo de confianza y comparar con la teoría con rigor.